# Device Agent Test Notebook

This notebook tests the `smol_device_agent.py` and `device_tools.py` implementation, including:
- Direct tool function testing
- Agent interaction testing
- Error handling
- Full workflow testing (draft -> submit)
- Edge cases and validation


In [1]:
# Setup: Import dependencies and load environment
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

# Add project root to Python path (works in Jupyter notebooks)
# Strategy: Walk up the directory tree until we find the project root (where 'src' directory exists)
current_dir = Path.cwd().resolve()
project_root = current_dir

# Walk up the directory tree to find project root
max_levels = 5  # Prevent infinite loops
for _ in range(max_levels):
    if (project_root / 'src').exists() and (project_root / 'src' / 'agents').exists():
        # Found project root (has src/agents directory)
        break
    parent = project_root.parent
    if parent == project_root:
        # Reached filesystem root
        break
    project_root = parent
else:
    # If we didn't break, use current directory as fallback
    project_root = current_dir

# Add to path if not already there
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"✓ Added project root to path: {project_root}")
else:
    print(f"✓ Project root already in path: {project_root}")

# Verify we can find the src directory
if not (project_root / 'src').exists():
    print(f"⚠ WARNING: Could not find 'src' directory in {project_root}")
    print(f"   Current working directory: {current_dir}")

# Load environment variables (try both project root and current dir)
env_file = project_root / '.env'
if env_file.exists():
    load_dotenv(env_file, override=True)
else:
    load_dotenv(override=True)

# Verify API key is set
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print(f"✓ OpenAI API Key loaded (starts with: {api_key[:8]}...)")
else:
    print("✗ WARNING: OPENAI_API_KEY not set in environment")


✓ Added project root to path: C:\projects\simpliAsk\simpliAsk
✓ OpenAI API Key loaded (starts with: sk-proj-...)


In [2]:
# Import the agent and tools
from src.agents.smol_device_agent import device_smol_agent
from src.tools import device_tools as dt

print("✓ Device agent and tools imported successfully")
print(f"✓ Agent name: {device_smol_agent.name}")
print(f"✓ Number of tools: {len(device_smol_agent.tools)}")

# Display tool names (handle different tool representations)
tool_names = []
for tool in device_smol_agent.tools:
    if hasattr(tool, 'name'):
        tool_names.append(tool.name)
    elif hasattr(tool, '__name__'):
        tool_names.append(tool.__name__)
    elif isinstance(tool, str):
        tool_names.append(tool)
    else:
        tool_names.append(str(tool))
print(f"✓ Tools: {tool_names}")


✓ Device agent and tools imported successfully
✓ Agent name: device_specialist
✓ Number of tools: 7
✓ Tools: ['get_available_devices_tool', 'draft_device_request_tool', 'submit_draft_device_request_tool', 'submit_device_request_tool', 'list_device_drafts_tool', 'check_device_request_status_tool', 'final_answer']


## 1. Direct Tool Testing

Test the underlying tool functions directly before testing the agent.


### Test 1.1: Get Available Devices


In [3]:
# Test 1.1: Get available devices
result = dt.get_available_devices()
devices_data = json.loads(result)
print("Available Devices:")
print(json.dumps(devices_data, indent=2))
print(f"\n✓ Total devices available: {len(devices_data)}")


Available Devices:
[
  {
    "id": 1,
    "name": "2M HDMI Cable",
    "cost": 6.5
  },
  {
    "id": 2,
    "name": "Wireless Mouse",
    "cost": 15.0
  },
  {
    "id": 3,
    "name": "Mechanical Keyboard",
    "cost": 45.0
  },
  {
    "id": 4,
    "name": "27-inch Monitor",
    "cost": 230.0
  },
  {
    "id": 5,
    "name": "USB-C Hub",
    "cost": 25.5
  },
  {
    "id": 6,
    "name": "External Hard Drive 1TB",
    "cost": 65.0
  },
  {
    "id": 7,
    "name": "Laptop Stand",
    "cost": 30.0
  },
  {
    "id": 8,
    "name": "Webcam 1080p",
    "cost": 40.0
  }
]

✓ Total devices available: 8


### Test 1.2: Create Draft Device Request


In [4]:
# Test 1.2: Create a draft device request
employee_id = "mark_tan"
device_id = 3  # Mechanical Keyboard
device_name = "Mechanical Keyboard"

draft_result = dt.draft_device_request(
    employee_id=employee_id,
    device_id=device_id,
    device_name=device_name
)

draft_data = json.loads(draft_result)
print("Draft Device Request Result:")
print(json.dumps(draft_data, indent=2))

# Extract draft_id for later tests
if "draft_id" in draft_data:
    draft_id = draft_data["draft_id"]
    print(f"\n✓ Draft created with draft_id: {draft_id}")
else:
    print("\n✗ Error: No draft_id in response")
    draft_id = None


--- SYSTEM: Draft created for mark_tan (device request) for Mechanical Keyboard ($45.00) ---
Draft Device Request Result:
{
  "status": "draft",
  "message": "Draft device request created.",
  "draft_id": "draft-90845",
  "draft": {
    "draft_id": "draft-90845",
    "employee_id": "mark_tan",
    "device_id": 3,
    "device_name": "Mechanical Keyboard",
    "device_cost": 45.0,
    "status": "draft"
  }
}

✓ Draft created with draft_id: draft-90845


### Test 1.3: List Device Drafts


In [5]:
# Test 1.3: List all drafts for the employee
result = dt.list_device_drafts(employee_id)
drafts_data = json.loads(result)
print(f"Total drafts for {employee_id}: {len(drafts_data.get('drafts', []))}")
print("\nDrafts list:")
print(json.dumps(drafts_data, indent=2))


Total drafts for mark_tan: 1

Drafts list:
{
  "employee_id": "mark_tan",
  "drafts": [
    {
      "draft_id": "draft-90845",
      "employee_id": "mark_tan",
      "device_id": 3,
      "device_name": "Mechanical Keyboard",
      "device_cost": 45.0,
      "status": "draft"
    }
  ]
}


### Test 1.4: Submit Draft Device Request


In [6]:
# Test 1.4: Submit the draft device request
if draft_id:
    submit_result = dt.submit_draft_device_request(employee_id, draft_id)
    submit_data = json.loads(submit_result)
    print("Submit Draft Result:")
    print(json.dumps(submit_data, indent=2))
    
    # Extract request_id for later tests
    if "request_id" in submit_data:
        request_id = submit_data["request_id"]
        print(f"\n✓ Request submitted with request_id: {request_id}")
    else:
        print("\n✗ Error: No request_id in response")
        request_id = None
else:
    print("✗ Skipping: No draft_id available")
    request_id = None


--- SYSTEM: Submitting draft draft-90845 as MW42176 - Mechanical Keyboard ($45.00) for mark_tan (status: pending) ---
Submit Draft Result:
{
  "status": "success",
  "request_id": "MW42176",
  "message": "Request ID #MW42176 submitted pending manager review."
}

✓ Request submitted with request_id: MW42176


### Test 1.5: Check Device Request Status


In [7]:
# Test 1.5: Check the status of the submitted device request
if request_id:
    status_result = dt.check_device_request_status(request_id)
    status_data = json.loads(status_result)
    print("Device Request Status Result:")
    print(json.dumps(status_data, indent=2))
else:
    print("✗ Skipping: No request_id available")


Device Request Status Result:
{
  "request_id": "MW42176",
  "status": "pending",
  "employee_id": "mark_tan",
  "device_id": 3,
  "device_name": "Mechanical Keyboard",
  "device_cost": 45.0
}


### Test 1.6: Direct Submit (Alternative Method)


In [8]:
# Test 1.6: Direct submit without draft (alternative method)
direct_submit_result = dt.submit_device_request(
    employee_id=employee_id,
    device_id=2,  # Wireless Mouse
    device_name="Wireless Mouse"
)

direct_submit_data = json.loads(direct_submit_result)
print("Direct Submit Result:")
print(json.dumps(direct_submit_data, indent=2))

if "request_id" in direct_submit_data:
    direct_request_id = direct_submit_data["request_id"]
    print(f"\n✓ Direct request submitted with request_id: {direct_request_id}")
else:
    direct_request_id = None


--- SYSTEM: Submitting request for 2: Wireless Mouse ($15.00) for mark_tan with request ID MW79107 ---
Direct Submit Result:
{
  "status": "success",
  "request_id": "MW79107",
  "message": "Request ID #MW79107 submitted pending manager review."
}

✓ Direct request submitted with request_id: MW79107


## 2. Error Handling Tests

Test various error cases and edge conditions.


### Test 2.1: Invalid Device ID


In [9]:
# Test 2.1: Invalid device ID
error_result = dt.draft_device_request(employee_id, 999, "Non-existent Device")
print("Error Test (invalid device ID):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (invalid device ID):
{
  "error": "Device with ID 999 not found"
}


### Test 2.2: Device Name Mismatch


In [10]:
# Test 2.2: Device name mismatch
error_result = dt.draft_device_request(employee_id, 1, "Wrong Device Name")
print("Error Test (device name mismatch):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (device name mismatch):
{
  "error": "Device name mismatch. Expected '2M HDMI Cable', got 'Wrong Device Name'"
}


### Test 2.3: Empty Employee ID


In [11]:
# Test 2.3: Empty employee ID
error_result = dt.draft_device_request("", 1, "2M HDMI Cable")
print("Error Test (empty employee ID):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (empty employee ID):
{
  "error": "Employee ID cannot be empty"
}


### Test 2.4: Invalid Request ID (Status Check)


In [12]:
# Test 2.4: Invalid request ID for status check
error_result = dt.check_device_request_status("MW99999")
print("Error Test (invalid request ID):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (invalid request ID):
{
  "error": "Request not found",
  "request_id": "MW99999"
}


### Test 2.5: Invalid Draft ID (Submit Draft)


In [13]:
# Test 2.5: Invalid draft ID for submission
error_result = dt.submit_draft_device_request(employee_id, "draft-99999")
print("Error Test (invalid draft ID):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (invalid draft ID):
{
  "error": "Draft not found"
}


### Test 2.6: Negative Device ID


In [14]:
# Test 2.6: Negative device ID
error_result = dt.submit_device_request(employee_id, -1, "Invalid Device")
print("Error Test (negative device ID):")
print(json.dumps(json.loads(error_result), indent=2))


Error Test (negative device ID):
{
  "error": "Device ID must be a positive integer"
}


## 3. Agent Interaction Tests

Test the agent's ability to handle user queries and use tools appropriately.


### Test 3.1: Agent - List Available Devices


In [15]:
# Test 3.1: Agent interaction - List available devices
query = "What devices are available for request?"
print(f"User Query: {query}\n")
print("Agent Response:")
response = device_smol_agent.run(query)
print(response)


User Query: What devices are available for request?

Agent Response:


╭────────────────────────────────────────── New run - device_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ What devices are available for request?                                                                         │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_available_devices_tool' with arguments: {}                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: |
  {
    "id": 1,
    "name": "2M HDMI Cable",
    "cost": 6.5
  },
  {
    "id": 2,
    "name": "Wireless Mouse",
    "cost": 15.0
  },
  {
    "id": 3,
    "name": "Mechanical Keyboard",
    "cost": 45.0
  },
  {
    "id": 4,
    "name": "27-inch Monitor",
    "cost": 230.0
  },
  {
    "id": 5,
    "name": "USB-C Hub",
    "cost": 25.5
  },
  {
    "id": 6,
    "name": "External Hard Drive 1TB",
    "cost": 65.0
  },
  {
    "id": 7,
    "name": "Laptop Stand",
    "cost": 30.0
  },
  {
    "id": 8,
    "name": "Webcam 1080p",
    "cost": 40.0
  }
]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The devices available for request are:\n1. 2M HDMI     │
│ Cable (ID: 1) - $6.5\n2. Wireless Mouse (ID: 2) - $15.0\n3. Mechanical Keyboard (ID: 3) - $45.0\n4. 27-inch     │
│ Monitor (ID: 4) - $230.0\n5. USB-C Hub (ID: 5) - $25.5\n6. External Hard Drive 1TB (ID: 6) - $65.0\n7. Laptop   │
│ Stand (ID: 7) - $30.0\n8. Webcam 1080p (ID: 8) - $40.0'}                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The devices available for request are:
1. 2M HDMI Cable (ID: 1) - $6.5
2. Wireless Mouse (ID: 2) - $15.0
3. Mechanical Keyboard (ID: 3) - $45.0
4. 27-inch Monitor (ID: 4) - $230.0
5. USB-C Hub (ID: 5) - $25.5
6. External Hard Drive 1TB (ID: 6) - $65.0
7. Laptop Stand (ID: 7) - $30.0
8. Webcam 1080p (ID: 8) - $40.0

Final answer: The devices available for request are:
1. 2M HDMI Cable (ID: 1) - $6.5
2. Wireless Mouse (ID: 2) - $15.0
3. Mechanical Keyboard (ID: 3) - $45.0
4. 27-inch Monitor (ID: 4) - $230.0
5. USB-C Hub (ID: 5) - $25.5
6. External Hard Drive 1TB (ID: 6) - $65.0
7. Laptop Stand (ID: 7) - $30.0
8. Webcam 1080p (ID: 8) - $40.0

[Step 2: Duration 2.73 seconds| Input tokens: 5,141 | Output tokens: 173]

The devices available for request are:
1. 2M HDMI Cable (ID: 1) - $6.5
2. Wireless Mouse (ID: 2) - $15.0
3. Mechanical Keyboard (ID: 3) - $45.0
4. 27-inch Monitor (ID: 4) - $230.0
5. USB-C Hub (ID: 5) - $25.5
6. External Hard Drive 1TB (ID: 6) - $65.0
7. Laptop Stand (ID: 7) - $30.0
8. Webcam 1080p (ID: 8) - $40.0


### Test 3.2: Agent - Create Draft Request


In [16]:
# Test 3.2: Agent interaction - Create draft request
query = "I would like to request a 27-inch Monitor. Please create a draft for me."
print(f"User Query: {query}\n")
print("Agent Response:")
response = device_smol_agent.run(query)
print(response)


User Query: I would like to request a 27-inch Monitor. Please create a draft for me.

Agent Response:


╭────────────────────────────────────────── New run - device_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ I would like to request a 27-inch Monitor. Please create a draft for me.                                        │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_available_devices_tool' with arguments: {}                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: |
  {
    "id": 1,
    "name": "2M HDMI Cable",
    "cost": 6.5
  },
  {
    "id": 2,
    "name": "Wireless Mouse",
    "cost": 15.0
  },
  {
    "id": 3,
    "name": "Mechanical Keyboard",
    "cost": 45.0
  },
  {
    "id": 4,
    "name": "27-inch Monitor",
    "cost": 230.0
  },
  {
    "id": 5,
    "name": "USB-C Hub",
    "cost": 25.5
  },
  {
    "id": 6,
    "name": "External Hard Drive 1TB",
    "cost": 65.0
  },
  {
    "id": 7,
    "name": "Laptop Stand",
    "cost": 30.0
  },
  {
    "id": 8,
    "name": "Webcam 1080p",
    "cost": 40.0
  }
]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Draft device request for a 27-inch Monitor has been    │
│ created successfully. The draft ID is draft-59722 and the device cost is $230. You can now review this draft or │
│ let me know if you want to submit it.'}                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Draft device request for a 27-inch Monitor has been created successfully. The draft ID is draft-59722
and the device cost is $230. You can now review this draft or let me know if you want to submit it.

Final answer: Draft device request for a 27-inch Monitor has been created successfully. The draft ID is draft-59722
and the device cost is $230. You can now review this draft or let me know if you want to submit it.

[Step 3: Duration 1.39 seconds| Input tokens: 8,057 | Output tokens: 104]

Draft device request for a 27-inch Monitor has been created successfully. The draft ID is draft-59722 and the device cost is $230. You can now review this draft or let me know if you want to submit it.


### Test 3.3: Agent - Check Request Status


In [17]:
# Test 3.3: Agent interaction - Check request status
if 'request_id' in locals() and request_id:
    query = f"What is the status of my device request {request_id}?"
    print(f"User Query: {query}\n")
    print("Agent Response:")
    response = device_smol_agent.run(query)
    print(response)
else:
    print("⚠ Skipping: No request_id available from previous tests")


User Query: What is the status of my device request MW42176?

Agent Response:


╭────────────────────────────────────────── New run - device_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ What is the status of my device request MW42176?                                                                │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'check_device_request_status_tool' with arguments: {'request_id': 'MW42176'}                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"request_id": "MW42176", "status": "pending", "employee_id": "mark_tan", "device_id": 3, 
"device_name": "Mechanical Keyboard", "device_cost": 45.0}

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The status of your device request MW42176 is pending.  │
│ The request is for a Mechanical Keyboard with a cost of $45.00.'}                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The status of your device request MW42176 is pending. The request is for a Mechanical Keyboard with a
cost of $45.00.

Final answer: The status of your device request MW42176 is pending. The request is for a Mechanical Keyboard with a
cost of $45.00.

[Step 2: Duration 0.98 seconds| Input tokens: 4,972 | Output tokens: 61]

The status of your device request MW42176 is pending. The request is for a Mechanical Keyboard with a cost of $45.00.


### Test 3.4: Agent - List My Drafts


In [18]:
# Test 3.4: Agent interaction - List drafts
query = "Show me all my draft device requests"
print(f"User Query: {query}\n")
print("Agent Response:")
response = device_smol_agent.run(query)
print(response)


User Query: Show me all my draft device requests

Agent Response:


╭────────────────────────────────────────── New run - device_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ Show me all my draft device requests                                                                            │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'list_device_drafts_tool' with arguments: {'employee_id': 'mark_tan'}                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {"employee_id": "mark_tan", "drafts": |{"draft_id": "draft-90845", "employee_id": "mark_tan", 
"device_id": 3, "device_name": "Mechanical Keyboard", "device_cost": 45.0, "status": "submitted"}, {"draft_id": 
"draft-59722", "employee_id": "mark_tan", "device_id": 4, "device_name": "27-inch Monitor", "device_cost": 230.0, 
"status": "draft"}]}

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'You have two draft device requests. One is a submitted │
│ draft for a Mechanical Keyboard with draft ID draft-90845 and costs $45. The other is an active draft for a     │
│ 27-inch Monitor with draft ID draft-59722 and costs $230.'}                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: You have two draft device requests. One is a submitted draft for a Mechanical Keyboard with draft ID 
draft-90845 and costs $45. The other is an active draft for a 27-inch Monitor with draft ID draft-59722 and costs 
$230.

Final answer: You have two draft device requests. One is a submitted draft for a Mechanical Keyboard with draft ID 
draft-90845 and costs $45. The other is an active draft for a 27-inch Monitor with draft ID draft-59722 and costs 
$230.

[Step 2: Duration 1.38 seconds| Input tokens: 5,025 | Output tokens: 85]

You have two draft device requests. One is a submitted draft for a Mechanical Keyboard with draft ID draft-90845 and costs $45. The other is an active draft for a 27-inch Monitor with draft ID draft-59722 and costs $230.


## 4. Full Workflow Tests

Test complete workflows from start to finish.


### Test 4.1: Complete Workflow - Draft to Submit


In [19]:
# Test 4.1: Complete workflow - Draft to Submit
print("=" * 60)
print("COMPLETE WORKFLOW TEST: Draft to Submit")
print("=" * 60)

# Step 1: User asks about devices
print("\n[Step 1] User: What devices can I request?")
response1 = device_smol_agent.run("What devices can I request?")
print(f"Agent: {response1}\n")

# Step 2: User requests a specific device
print("[Step 2] User: I want to request a USB-C Hub")
response2 = device_smol_agent.run("I want to request a USB-C Hub")
print(f"Agent: {response2}\n")

# Extract draft_id from response if available (this is a simplified test)
# In a real scenario, the agent would present the draft and wait for confirmation

# Step 3: User confirms and wants to submit
print("[Step 3] User: Yes, please submit my draft")
response3 = device_smol_agent.run("Yes, please submit my draft")
print(f"Agent: {response3}\n")

print("=" * 60)


COMPLETE WORKFLOW TEST: Draft to Submit

[Step 1] User: What devices can I request?


╭────────────────────────────────────────── New run - device_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ What devices can I request?                                                                                     │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_available_devices_tool' with arguments: {}                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: |
  {
    "id": 1,
    "name": "2M HDMI Cable",
    "cost": 6.5
  },
  {
    "id": 2,
    "name": "Wireless Mouse",
    "cost": 15.0
  },
  {
    "id": 3,
    "name": "Mechanical Keyboard",
    "cost": 45.0
  },
  {
    "id": 4,
    "name": "27-inch Monitor",
    "cost": 230.0
  },
  {
    "id": 5,
    "name": "USB-C Hub",
    "cost": 25.5
  },
  {
    "id": 6,
    "name": "External Hard Drive 1TB",
    "cost": 65.0
  },
  {
    "id": 7,
    "name": "Laptop Stand",
    "cost": 30.0
  },
  {
    "id": 8,
    "name": "Webcam 1080p",
    "cost": 40.0
  }
]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Your device request draft with draft ID draft-59722    │
│ for the 27-inch Monitor has been successfully submitted. The request ID is MW17762 and is pending manager       │
│ review.'}                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Your device request draft with draft ID draft-59722 for the 27-inch Monitor has been successfully 
submitted. The request ID is MW17762 and is pending manager review.

Final answer: Your device request draft with draft ID draft-59722 for the 27-inch Monitor has been successfully 
submitted. The request ID is MW17762 and is pending manager review.

[Step 3: Duration 1.00 seconds| Input tokens: 7,846 | Output tokens: 97]

Agent: Your device request draft with draft ID draft-59722 for the 27-inch Monitor has been successfully submitted. The request ID is MW17762 and is pending manager review.



### Test 4.2: Direct Submit Workflow


In [20]:
# Test 4.2: Direct submit workflow (alternative method)
print("=" * 60)
print("DIRECT SUBMIT WORKFLOW TEST")
print("=" * 60)

print("\n[Step 1] User: I need a Webcam 1080p immediately, submit it directly without a draft")
response = device_smol_agent.run("I need a Webcam 1080p immediately, submit it directly without a draft")
print(f"Agent: {response}\n")

print("=" * 60)


DIRECT SUBMIT WORKFLOW TEST

[Step 1] User: I need a Webcam 1080p immediately, submit it directly without a draft


╭────────────────────────────────────────── New run - device_specialist ──────────────────────────────────────────╮
│                                                                                                                 │
│ I need a Webcam 1080p immediately, submit it directly without a draft                                           │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_available_devices_tool' with arguments: {}                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: |
  {
    "id": 1,
    "name": "2M HDMI Cable",
    "cost": 6.5
  },
  {
    "id": 2,
    "name": "Wireless Mouse",
    "cost": 15.0
  },
  {
    "id": 3,
    "name": "Mechanical Keyboard",
    "cost": 45.0
  },
  {
    "id": 4,
    "name": "27-inch Monitor",
    "cost": 230.0
  },
  {
    "id": 5,
    "name": "USB-C Hub",
    "cost": 25.5
  },
  {
    "id": 6,
    "name": "External Hard Drive 1TB",
    "cost": 65.0
  },
  {
    "id": 7,
    "name": "Laptop Stand",
    "cost": 30.0
  },
  {
    "id": 8,
    "name": "Webcam 1080p",
    "cost": 40.0
  }
]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Your device request for the Webcam 1080p has been      │
│ successfully submitted directly without a draft. The request ID is MW85352. It is currently pending manager     │
│ review.'}                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Your device request for the Webcam 1080p has been successfully submitted directly without a draft. 
The request ID is MW85352. It is currently pending manager review.

Final answer: Your device request for the Webcam 1080p has been successfully submitted directly without a draft. 
The request ID is MW85352. It is currently pending manager review.

[Step 3: Duration 1.22 seconds| Input tokens: 8,003 | Output tokens: 93]

Agent: Your device request for the Webcam 1080p has been successfully submitted directly without a draft. The request ID is MW85352. It is currently pending manager review.



## 5. Edge Cases and Validation

Test edge cases and boundary conditions.


### Test 5.1: Multiple Drafts


In [21]:
# Test 5.1: Create multiple drafts
print("Creating multiple drafts for testing...\n")

# Create draft 1
draft1 = dt.draft_device_request(employee_id, 4, "27-inch Monitor")
draft1_data = json.loads(draft1)
print(f"Draft 1: {draft1_data.get('draft_id', 'N/A')}")

# Create draft 2
draft2 = dt.draft_device_request(employee_id, 6, "External Hard Drive 1TB")
draft2_data = json.loads(draft2)
print(f"Draft 2: {draft2_data.get('draft_id', 'N/A')}")

# Create draft 3
draft3 = dt.draft_device_request(employee_id, 7, "Laptop Stand")
draft3_data = json.loads(draft3)
print(f"Draft 3: {draft3_data.get('draft_id', 'N/A')}")

# List all drafts
print("\nAll drafts:")
all_drafts = dt.list_device_drafts(employee_id)
print(json.dumps(json.loads(all_drafts), indent=2))


Creating multiple drafts for testing...

--- SYSTEM: Draft created for mark_tan (device request) for 27-inch Monitor ($230.00) ---
Draft 1: draft-38053
--- SYSTEM: Draft created for mark_tan (device request) for External Hard Drive 1TB ($65.00) ---
Draft 2: draft-67014
--- SYSTEM: Draft created for mark_tan (device request) for Laptop Stand ($30.00) ---
Draft 3: draft-37419

All drafts:
{
  "employee_id": "mark_tan",
  "drafts": [
    {
      "draft_id": "draft-90845",
      "employee_id": "mark_tan",
      "device_id": 3,
      "device_name": "Mechanical Keyboard",
      "device_cost": 45.0,
      "status": "submitted"
    },
    {
      "draft_id": "draft-59722",
      "employee_id": "mark_tan",
      "device_id": 4,
      "device_name": "27-inch Monitor",
      "device_cost": 230.0,
      "status": "submitted"
    },
    {
      "draft_id": "draft-52540",
      "employee_id": "mark_tan",
      "device_id": 5,
      "device_name": "USB-C Hub",
      "device_cost": 25.5,
      "status

### Test 5.2: Submit Already Submitted Draft


In [22]:
# Test 5.2: Try to submit an already submitted draft
if 'draft_id' in locals() and draft_id:
    # First submit it
    first_submit = dt.submit_draft_device_request(employee_id, draft_id)
    print("First submission:")
    print(json.dumps(json.loads(first_submit), indent=2))
    
    # Try to submit again
    print("\nAttempting to submit the same draft again:")
    second_submit = dt.submit_draft_device_request(employee_id, draft_id)
    print(json.dumps(json.loads(second_submit), indent=2))
else:
    print("⚠ Skipping: No draft_id available")


First submission:
{
  "error": "Draft is already submitted"
}

Attempting to submit the same draft again:
{
  "error": "Draft is already submitted"
}


### Test 5.3: Wrong Employee ID for Draft Submission


In [23]:
# Test 5.3: Try to submit a draft with wrong employee ID
# First create a draft
test_draft = dt.draft_device_request(employee_id, 8, "Webcam 1080p")
test_draft_data = json.loads(test_draft)
test_draft_id = test_draft_data.get("draft_id")

if test_draft_id:
    # Try to submit with wrong employee ID
    wrong_employee_result = dt.submit_draft_device_request("wrong_employee", test_draft_id)
    print("Error Test (wrong employee ID):")
    print(json.dumps(json.loads(wrong_employee_result), indent=2))
else:
    print("⚠ Could not create test draft")


--- SYSTEM: Draft created for mark_tan (device request) for Webcam 1080p ($40.00) ---
Error Test (wrong employee ID):
{
  "error": "Draft does not belong to this employee"
}


## 6. Summary and Cleanup

Summary of test results and cleanup notes.


In [24]:
# Summary: Display all stored requests and drafts
print("=" * 60)
print("TEST SUMMARY")
print("=" * 60)

# Note: In a real scenario, you might want to clear test data
# For now, we'll just show what was created

print("\n✓ All tests completed successfully!")
print("\nNote: This is a prototype implementation using in-memory storage.")
print("In a production environment, you would:")
print("  - Use a persistent database")
print("  - Implement proper cleanup between test runs")
print("  - Add more comprehensive error handling")
print("  - Implement request status updates (approved/rejected)")
print("\n" + "=" * 60)


TEST SUMMARY

✓ All tests completed successfully!

Note: This is a prototype implementation using in-memory storage.
In a production environment, you would:
  - Use a persistent database
  - Implement proper cleanup between test runs
  - Add more comprehensive error handling
  - Implement request status updates (approved/rejected)

